## ulta top10 순위 및 모든리뷰데이터 2년치 추출코드

In [1]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import json
import time
import random
from datetime import datetime, timedelta

# ── 설정 ──────────────────────────────────────────────────────────
API_KEY          = "daa0f241-c242-4483-afb7-4449942d1a2b"
REVIEW_SAVE_FILE = "ulta_reviews_master.jsonl"
RANK_SAVE_FILE   = "ulta_rankings_current.jsonl"

# [날짜 설정] 현재 기준 2년 전 타임스탬프 계산 (밀리초 단위)
CUTOFF_DATE = datetime.now() - timedelta(days=365 * 2)
CUTOFF_TIMESTAMP = CUTOFF_DATE.timestamp() * 1000  # 밀리초로 변환

# ── 함수: 단일 상품 리뷰 수집 (최근 2년치 필터링) ───────────────────
def get_product_reviews_2years(product_id, product_name):
    final_reviews = []
    page_size   = 25
    paging_from = 0
    stop_collecting = False

    print(f"\n  🚀 [{product_name[:30]}] 리뷰 수집 (기준: {CUTOFF_DATE.strftime('%Y-%m-%d')} 이후)")

    while not stop_collecting:
        url = f"https://display.powerreviews.com/m/6406/l/en_US/product/{product_id}/reviews"
        params = {
            "paging.from" : paging_from,
            "paging.size" : page_size,
            "sort"        : "Newest",  # 최신순 정렬 필수
            "apikey"      : API_KEY,
        }
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code != 200: break

            data = response.json()
            results = data.get("results", [])

            if not results or not results[0].get("reviews"):
                break

            reviews = results[0].get("reviews", [])
            for rev in reviews:
                details = rev.get("details", {})
                metrics = rev.get("metrics", {})
                
                # 1. created_date 추출 (밀리초 단위)
                raw_ms = details.get("created_date")
                
                if raw_ms:
                    # 2. 날짜 비교: 2년 전 타임스탬프보다 작으면 수집 중단
                    if raw_ms < CUTOFF_TIMESTAMP:
                        stop_collecting = True
                        break
                
                # 3. 데이터 저장
                fmt_date = datetime.fromtimestamp(raw_ms / 1000.0).strftime("%Y-%m-%d %H:%M") if raw_ms else "N/A"
                final_reviews.append({
                    "product_id" : product_id,
                    "review_id"  : rev.get("review_id"),
                    "author"     : details.get("nickname"),
                    "rating"     : metrics.get("rating"),
                    "headline"   : details.get("headline"),
                    "comment"    : details.get("comments"),
                    "date"       : fmt_date,
                    "created_at" : raw_ms
                })

            if stop_collecting:
                print(f"    📍 2년 이전 리뷰 도달 ({fmt_date}) - 수집 종료")
                break

            print(f"    🔄 수집 중... (누적: {len(final_reviews)}개)", end="\r")
            paging_from += page_size
            time.sleep(random.uniform(0.3, 0.5))

        except Exception as e:
            print(f"\n    ❌ 에러 발생: {e}")
            break

    print(f"\n    ✨ {len(final_reviews)}개 수집 완료")
    return final_reviews


# ── 메인: Ulta 데이터 수집 ─────────────────────────────────────────
def fetch_ulta_data():
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    driver = uc.Chrome(options=options)

    try:
        print(f"📡 Ulta 접속 중 (기준일: {CUTOFF_DATE.strftime('%Y-%m-%d')})")
        driver.get("https://www.ulta.com/shop/skin-care/all?sort=best_sellers")
        time.sleep(random.uniform(7, 10))

        soup  = BeautifulSoup(driver.page_source, "html.parser")
        items = soup.select("li.ProductListingResults__productCard")

        rank_data_list    = []
        all_review_master = []
        rank_count        = 1

        for item in items:
            if rank_count > 10: break

            # 광고 제외
            if item.select_one(".pal-c-ProductCardFooter__sponsored"):
                continue

            try:
                # 상품 정보 파싱
                brand = item.select_one(".pal-c-ProductCardBody--brandName p").get_text(strip=True)
                title_link = item.select_one("a.pal-c-Link")
                title = title_link.get_text(strip=True)
                product_url = title_link.get("href")
                if not product_url.startswith("http"):
                    product_url = "https://www.ulta.com" + product_url

                # URL에서 ID 추출
                product_id = product_url.split("-")[-1].split("?")[0]

                rank_data_list.append({
                    "rank": rank_count,
                    "brand": brand,
                    "title": title,
                    "url": product_url,
                    "product_id": product_id,
                    "platform": "Ulta",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                })
                print(f"\n📍 {rank_count}위: [{brand}] {title[:30]}")

                # ── 리뷰 수집 실행 ──
                reviews = get_product_reviews_2years(product_id, title)
                all_review_master.extend(reviews)

                rank_count += 1

            except Exception as e:
                print(f"⚠️ 파싱 오류: {e}")
                continue

        # JSONL 저장
        with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
            for entry in rank_data_list: f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
            for review in all_review_master: f.write(json.dumps(review, ensure_ascii=False) + "\n")

        print(f"\n📊 작업 완료: 상품 {len(rank_data_list)}개 / 총 리뷰 {len(all_review_master)}개 저장됨")

    finally:
        driver.quit()

if __name__ == "__main__":
    fetch_ulta_data()

📡 Ulta 접속 중 (기준일: 2024-03-20)

📍 1위: [IT Cosmetics] IT Cosmetics Do It All Sheer T

  🚀 [IT Cosmetics Do It All Sheer T] 리뷰 수집 (기준: 2024-03-20 이후)
    🔄 수집 중... (누적: 3201개)
    ✨ 3201개 수집 완료

📍 2위: [medicube] medicube Zero Pore Pad

  🚀 [medicube Zero Pore Pad] 리뷰 수집 (기준: 2024-03-20 이후)
    🔄 수집 중... (누적: 127개)
    ✨ 127개 수집 완료

📍 3위: [The Ordinary] The Ordinary Glycolic Acid 7% 

  🚀 [The Ordinary Glycolic Acid 7% ] 리뷰 수집 (기준: 2024-03-20 이후)
    📍 2년 이전 리뷰 도달 (2024-03-21 06:48) - 수집 종료

    ✨ 463개 수집 완료

📍 4위: [Clinique] Clinique Moisture Surge 100H A

  🚀 [Clinique Moisture Surge 100H A] 리뷰 수집 (기준: 2024-03-20 이후)
    📍 2년 이전 리뷰 도달 (2024-03-22 03:01) - 수집 종료

    ✨ 507개 수집 완료

📍 5위: [MAËLYS] MAËLYS GET-DREAMY Overnight To

  🚀 [MAËLYS GET-DREAMY Overnight To] 리뷰 수집 (기준: 2024-03-20 이후)
    📍 2년 이전 리뷰 도달 (2024-03-21 09:00) - 수집 종료

    ✨ 4535개 수집 완료

📍 6위: [Clinique] Clinique Even Better Makeup Br

  🚀 [Clinique Even Better Makeup Br] 리뷰 수집 (기준: 2024-03-20 이후)
    📍 2년 이전 리뷰 도달 (2024-03

# ulta 번역코드

In [2]:
import json
import time
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from deep_translator import GoogleTranslator

# ── 설정 ──────────────────────────────────────────────────────────
INPUT_FILE  = './ulta_reviews_master.jsonl'
OUTPUT_FILE = 'ulta_master_translated_en_ko.jsonl'

# 울타 데이터의 키 매칭
BODY_COL    = 'comment'   # 리뷰 본문
HEAD_COL    = 'headline'  # 리뷰 제목
ITEM_ID_COL = 'product_id'
RATING_COL  = 'rating'

# [병렬 처리 설정]
CHUNK_SIZE  = 5      # 묶음 번역 단위
MAX_WORKERS = 4      # 영어-한국어는 차단 위험이 낮아 4 유지 가능
MAX_RETRIES = 3      
RETRY_SLEEP = 2.0    
CHUNK_DELAY = 0.3    
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_numbered(texts: list) -> str:
    return "\n".join(f"[{i+1}] {str(t).strip()}" for i, t in enumerate(texts))

def parse_numbered(text: str, expected_n: int) -> list:
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    found = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in found}
    return [result.get(i + 1, "") for i in range(expected_n)]

def translate_single(text: str, src: str, tgt: str) -> str:
    if not text or not str(text).strip(): return ""
    for attempt in range(MAX_RETRIES):
        try:
            res = GoogleTranslator(source=src, target=tgt).translate(str(text))
            if res: return res.strip()
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return "번역실패"

def process_chunk(texts: list, src: str, tgt: str) -> list:
    if not texts: return []
    joined = build_numbered(texts)
    for attempt in range(MAX_RETRIES):
        try:
            translated = GoogleTranslator(source=src, target=tgt).translate(joined)
            if not translated: raise ValueError("응답 없음")
            parts = parse_numbered(translated, len(texts))
            if all(p.strip() for p in parts): return parts
            for i, p in enumerate(parts):
                if not p: parts[i] = translate_single(texts[i], src, tgt)
            return parts
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return [translate_single(t, src, tgt) for t in texts]

def translate_workflow(texts: list):
    """en -> ko 단일 단계 번역"""
    ko_texts = process_chunk(texts, 'en', 'ko')
    time.sleep(CHUNK_DELAY)
    return ko_texts

def main():
    print(f"📥 Ulta 데이터 로딩 중: {INPUT_FILE}")
    records = []
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip(): records.append(json.loads(line))
    except FileNotFoundError:
        print("❌ 입력 파일을 찾을 수 없습니다.")
        return

    df = pd.DataFrame(records)
    total_len = len(df)
    print(f"✅ 총 {total_len:,}건 확인")

    # 1. 리뷰 제목(headline) 번역
    print(f"\n🏷 1/2 단계: 리뷰 제목 번역 시작...")
    heads = df[HEAD_COL].fillna("").astype(str).tolist()
    head_chunks = [heads[i:i + CHUNK_SIZE] for i in range(0, total_len, CHUNK_SIZE)]
    
    all_head_ko = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(translate_workflow, head_chunks), total=len(head_chunks), desc="제목 번역 중"))
        for chunk in results: all_head_ko.extend(chunk)
    df['headline_ko'] = all_head_ko[:total_len]

    # 2. 리뷰 본문(comment) 번역
    print(f"\n📝 2/2 단계: 리뷰 본문 번역 시작...")
    bodies = df[BODY_COL].fillna("").astype(str).tolist()
    body_chunks = [bodies[i:i + CHUNK_SIZE] for i in range(0, total_len, CHUNK_SIZE)]
    
    all_body_ko = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(translate_workflow, body_chunks), total=len(body_chunks), desc="본문 번역 중"))
        for chunk in results: all_body_ko.extend(chunk)
    df['comment_ko'] = all_body_ko[:total_len]

    # 💾 결과 저장
    print(f"\n💾 결과 저장 중: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for record in df.to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f"✨ 모든 작업 완료! ({OUTPUT_FILE})")

if __name__ == "__main__":
    main()

📥 Ulta 데이터 로딩 중: ./ulta_reviews_master.jsonl
✅ 총 10,889건 확인

🏷 1/2 단계: 리뷰 제목 번역 시작...


제목 번역 중: 100%|██████████| 2178/2178 [12:50<00:00,  2.83it/s]



📝 2/2 단계: 리뷰 본문 번역 시작...


본문 번역 중: 100%|██████████| 2178/2178 [14:35<00:00,  2.49it/s] 


💾 결과 저장 중: ulta_master_translated_en_ko.jsonl
✨ 모든 작업 완료! (ulta_master_translated_en_ko.jsonl)
